<a href="https://colab.research.google.com/github/ecenur08/language-learning-assistant/blob/main/cevirichatbot2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install gradio==3.50.2 chromadb==0.4.24 sentence-transformers deep-translator langdetect --quiet
print("Kurulum tamam!")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/root/.kaggle', exist_ok=True)
!cp /content/kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json
print("Hazır!")

In [ ]:
with open('/content/drive/MyDrive/cevirichatbot/TR2EN.txt', 'r', encoding='utf-8') as f:
    satirlar = f.readlines()

print(f"Toplam satır: {len(satirlar)}")
print("İlk 3 satır:")
for s in satirlar[:3]:
    print(s.strip())

In [ ]:
import pandas as pd

ciftler = []
for satir in satirlar:
    parcalar = satir.strip().split('\t')
    if len(parcalar) >= 2:
        ciftler.append({'ingilizce': parcalar[0], 'turkce': parcalar[1]})

df = pd.DataFrame(ciftler)
print(f"Boyut: {df.shape}")
print(df.head(3))

In [ ]:
!pip install gradio chromadb sentence-transformers deep-translator langdetect --quiet
print("Tamam!")

In [ ]:
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

# 50.000 örnek alalım
sample = 50000
cumle = df['ingilizce'].tolist()[:sample] + df['turkce'].tolist()[:sample]
etiket = [0] * sample + [1] * sample

tokenizer = Tokenizer(num_words=20000, oov_token='<OOV>')
tokenizer.fit_on_texts(cumle)
X = pad_sequences(tokenizer.texts_to_sequences(cumle), maxlen=30)
y = np.array(etiket)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = tf.keras.Sequential([
    tf.keras.layers.Embedding(20000, 32, input_length=30),
    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

print("Model eğitiliyor...")
history = model.fit(X_train, y_train,
                    epochs=5,
                    batch_size=128,
                    validation_split=0.1,
                    verbose=1)

loss, acc = model.evaluate(X_test, y_test, verbose=0)
print(f"\nDil tespit doğruluğu: %{acc*100:.1f}")

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer

print("Embedding modeli yükleniyor...")
embedder = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

chroma_client = chromadb.Client()
collection = chroma_client.create_collection("ceviri_veritabani")

print("Cümleler vektör veritabanına ekleniyor...")
sample_df = df.sample(5000, random_state=42)

batch_size = 100
for i in range(0, len(sample_df), batch_size):
    batch = sample_df.iloc[i:i+batch_size]
    metinler = batch['ingilizce'].tolist()
    embeddings = embedder.encode(metinler).tolist()
    collection.add(
        documents=metinler,
        embeddings=embeddings,
        metadatas=[{"turkce": t} for t in batch['turkce'].tolist()],
        ids=[f"doc_{i+j}" for j in range(len(metinler))]
    )

print(f"RAG hazır! {collection.count()} cümle eklendi.")

In [ ]:
def ceviri_chatbot(mesaj, hedef_dil, gecmis):
    if len(mesaj.strip()) < 1:
        return gecmis + [(mesaj, "Lütfen bir şey yazın.")]

    # Dil tespiti
    kaynak_dil = dil_tespit_et(mesaj)
    kaynak_ad = DILLER.get(kaynak_dil, ('Bilinmiyor', '🌐'))

    # Hedef dil kodu
    hedef_kod = [k for k, v in DILLER.items() if v[0] == hedef_dil][0]
    hedef_ad = DILLER[hedef_kod]

    # Çeviri
    try:
        ceviri = GoogleTranslator(source=kaynak_dil, target=hedef_kod).translate(mesaj)
    except:
        ceviri = "Çeviri yapılamadı."

    # RAG — benzer cümleler getir
    benzerler = rag_benzer_bul(mesaj)

    # Sonraki cümle önerileri — RAG'dan gelen benzer cümlelerin devamını öner
    try:
        oneriler = []
        for b in benzerler[:3]:
            ingilizce_cumle = b.split('→')[0].replace('🇬🇧', '').strip()
            oneri_tr = GoogleTranslator(source='en', target=kaynak_dil).translate(ingilizce_cumle)
            oneri_hedef = GoogleTranslator(source=kaynak_dil, target=hedef_kod).translate(oneri_tr)
            oneriler.append(f"  💬 {oneri_tr}  →  {oneri_hedef}")
    except:
        oneriler = ["  💬 Öneri yüklenemedi."]

    oneri_metni = "\n".join(oneriler)

    yanit = f"""━━━━━━━━━━━━━━━━━━━━━
🔤  {kaynak_ad[1]} {kaynak_ad[0]}  →  {hedef_ad[1]} {hedef_ad[0]}
━━━━━━━━━━━━━━━━━━━━━

✅  **Çeviri:**
{ceviri}

━━━━━━━━━━━━━━━━━━━━━
💡  **Konuşmaya Devam Et:**
{oneri_metni}
━━━━━━━━━━━━━━━━━━━━━"""

    return gecmis + [(mesaj, yanit)]

print("Güncellendi!")

In [ ]:
!pip install groq --quiet
print("Tamam!")

In [ ]:
def ceviri_chatbot(mesaj, hedef_dil, gecmis):
    if len(mesaj.strip()) < 1:
        return gecmis + [(mesaj, "Lütfen bir şey yazın.")]

    kaynak_dil = dil_tespit_et(mesaj)
    kaynak_ad = DILLER.get(kaynak_dil, ('Bilinmiyor', '🌐'))
    hedef_kod = [k for k, v in DILLER.items() if v[0] == hedef_dil][0]
    hedef_ad = DILLER[hedef_kod]

    try:
        ceviri = GoogleTranslator(source=kaynak_dil, target=hedef_kod).translate(mesaj)
    except:
        ceviri = "Çeviri yapılamadı."

    benzerler = rag_benzer_bul(mesaj)

    try:
        llm_tavsiye = llm_cevap_uret(mesaj, ceviri, benzerler, kaynak_ad[0], hedef_ad[0])

        # LLM cümlesini RAG'a ekle
        yeni_embedding = embedder.encode([mesaj]).tolist()
        yeni_id = f"llm_{collection.count() + 1}"
        collection.add(
            documents=[mesaj],
            embeddings=yeni_embedding,
            metadatas=[{"turkce": ceviri, "kaynak": "llm"}],
            ids=[yeni_id]
        )
    except:
        llm_tavsiye = "Tavsiye yüklenemedi."

    yanit = f"""━━━━━━━━━━━━━━━━━━━━━
🔤  {kaynak_ad[1]} {kaynak_ad[0]}  →  {hedef_ad[1]} {hedef_ad[0]}
━━━━━━━━━━━━━━━━━━━━━

✅  **Çeviri:**
{ceviri}

━━━━━━━━━━━━━━━━━━━━━
🤖  **Konuşmaya Devam Et:**
{llm_tavsiye}
━━━━━━━━━━━━━━━━━━━━━"""

    return gecmis + [(mesaj, yanit)]

print("Güncellendi!")

In [ ]:
from groq import Groq

GROQ_API_KEY = "YOUR_GROQ_API_KEY"

groq_client = Groq(api_key=GROQ_API_KEY)

def llm_cevap_uret(mesaj, ceviri, benzerler, kaynak_dil, hedef_dil):
    benzer_metni = "\n".join(benzerler)

    prompt = f"""Sen bir dil öğrenme asistanısın.
Kullanıcı {kaynak_dil} dilinde şunu yazdı: "{mesaj}"
{hedef_dil} diline çevirisi: "{ceviri}"

Aşağıdaki benzer cümleleri kullanarak konuşmayı doğal şekilde devam ettirecek 3 cümle öner.
Benzer cümleler:
{benzer_metni}

Sadece 3 cümle yaz, her biri {kaynak_dil} ve {hedef_dil} dilinde olsun.
Format:
- [{kaynak_dil}] ... → [{hedef_dil}] ...
Türkçe açıklama ekleme, sadece cümleleri yaz."""

    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=200
    )
    return response.choices[0].message.content

print("LLM hazır!")

In [ ]:
import gradio as gr

css = """
* { box-sizing: border-box; }
footer { display: none !important; }
.gradio-container {
    background: linear-gradient(135deg, #1a1a2e 0%, #16213e 40%, #0f3460 70%, #533483 100%) !important;
    min-height: 100vh;
    padding: 20px !important;
}
.main-title {
    text-align: center;
    background: linear-gradient(90deg, #a78bfa, #60a5fa, #f472b6);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
    font-size: 2.8em;
    font-weight: 900;
    margin-bottom: 5px;
}
.sub-title {
    text-align: center;
    color: rgba(255,255,255,0.6);
    font-size: 1em;
    margin-bottom: 20px;
}
textarea {
    background: rgba(255,255,255,0.08) !important;
    border: 1px solid rgba(255,255,255,0.2) !important;
    border-radius: 14px !important;
    color: white !important;
}
textarea::placeholder { color: rgba(255,255,255,0.4) !important; }
button.primary {
    background: linear-gradient(135deg, #667eea 0%, #764ba2 100%) !important;
    border: none !important;
    border-radius: 14px !important;
    font-weight: 700 !important;
    color: white !important;
}
button.secondary {
    background: rgba(255,255,255,0.08) !important;
    border: 1px solid rgba(255,255,255,0.2) !important;
    color: rgba(255,255,255,0.7) !important;
    border-radius: 12px !important;
}
.info-panel {
    background: rgba(255,255,255,0.07) !important;
    border: 1px solid rgba(255,255,255,0.15) !important;
    border-radius: 20px !important;
    padding: 20px !important;
    color: white !important;
}
"""

with gr.Blocks(css=css, theme=gr.themes.Base()) as demo:

    gr.HTML("""
    <div class="main-title">📚 Dil Öğrenme Asistanı</div>
    <div class="sub-title">✨ Yapay Zeka ve RAG Destekli Çok Dilli Çeviri</div>
    """)

    with gr.Row():
        with gr.Column(scale=4):
            chatbot_ui = gr.Chatbot(
                label="",
                height=420,
                bubble_full_width=False,
                avatar_images=(
                    "https://cdn-icons-png.flaticon.com/512/1946/1946429.png",
                    "https://cdn-icons-png.flaticon.com/512/4712/4712109.png"
                )
            )
            with gr.Row():
                mesaj_kutusu = gr.Textbox(
                    placeholder="✍️ Herhangi bir dilde yazın...",
                    label="",
                    lines=2,
                    scale=4
                )
                hedef_dil = gr.Dropdown(
                    choices=['Türkçe', 'İngilizce', 'Almanca', 'Fransızca', 'İspanyolca', 'İtalyanca'],
                    value='İngilizce',
                    label="Hedef Dil",
                    scale=1
                )
                gonder = gr.Button("Çevir 🔄", variant="primary", scale=1)
            temizle = gr.Button("🗑️ Temizle", variant="secondary")

        with gr.Column(scale=1, min_width=200, elem_classes=["info-panel"]):
            gr.Markdown("""
            ### ⚙️ Nasıl Çalışır?
            **1.** Herhangi bir dilde yazın
            **2.** 🧠 Yapay zeka dili tespit eder
            **3.** Hedef dili seçin
            **4.** 🔄 Çeviri yapılır
            **5.** 🗄️ RAG benzer cümleler getirir

            ---

            ### 🌍 Desteklenen Diller
            🇹🇷 Türkçe
            🇬🇧 İngilizce
            🇩🇪 Almanca
            🇫🇷 Fransızca
            🇪🇸 İspanyolca
            🇮🇹 İtalyanca

            ---

            ### 📊 Model Bilgisi
            🧠 **Yapay Zeka**
            🎯 Doğruluk: **%99.7**
            📦 **473K** cümle
            🗄️ **ChromaDB** RAG
            """)

    gr.Examples(
        examples=[
            ["Merhaba, nasılsın?"],
            ["I love learning new languages."],
            ["Bonjour, comment allez-vous?"],
            ["Guten Morgen, wie geht es Ihnen?"],
            ["Buenos días, ¿cómo estás?"],
        ],
        inputs=mesaj_kutusu,
        label="💡 Örnek Cümleler"
    )

    gecmis_state = gr.State([])

    gonder.click(
        fn=ceviri_chatbot,
        inputs=[mesaj_kutusu, hedef_dil, gecmis_state],
        outputs=[chatbot_ui]
    ).then(lambda h: h, gecmis_state, gecmis_state)

    mesaj_kutusu.submit(
        fn=ceviri_chatbot,
        inputs=[mesaj_kutusu, hedef_dil, gecmis_state],
        outputs=[chatbot_ui]
    )

    temizle.click(lambda: ([], []), outputs=[chatbot_ui, gecmis_state])

demo.launch(share=True, debug=False)